# Preprocessing — Cartes spatiales SWI

Ce notebook transforme la base brute `jointure_meteo_swi_argile_nearest.gpkg`
en tenseur de cartes spatiales pret a l'emploi pour les notebooks d'entrainement.

**Sortie** : `data/maps/` contenant :
- `swi_maps.npy` — cartes SWI physiques `(T, H_crop, W_crop)`
- `swi_padded.npy` — cartes normalisees + paddees `(T, H_pad, W_pad)`
- `mask_padded.npy` — masque France `(H_pad, W_pad)`
- `metadata.json` — dimensions, stats mensuelles, dates


## 1. Imports et chargement

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import json, gc
from pathlib import Path
import matplotlib.pyplot as plt

OUT_DIR = Path('data/maps')
OUT_DIR.mkdir(parents=True, exist_ok=True)

gdf = gpd.read_file('data/processed/jointure_meteo_swi_argile_nearest.gpkg')
cols_f = gdf.select_dtypes('float64').columns
gdf[cols_f] = gdf[cols_f].astype('float32')
print(f'{len(gdf):,} lignes  |  {gdf["NUMERO"].nunique():,} points spatiaux')

## 2. Grille spatiale (arrondi au pas SAFRAN 8000 m)

In [ ]:
STEP = 8000
lambx_g = (gdf['LAMBX'] / STEP).round().astype(int) * STEP
lamby_g = (gdf['LAMBY'] / STEP).round().astype(int) * STEP

unique_x = np.sort(lambx_g.unique()).astype(np.int64)
unique_y = np.sort(lamby_g.unique()).astype(np.int64)
x_to_col = {int(v): i for i, v in enumerate(unique_x)}
y_to_row  = {int(v): i for i, v in enumerate(unique_y)}
H_full, W_full = len(unique_y), len(unique_x)
print(f'Grille complete : {H_full} x {W_full}')

## 3. Tenseur SWI complet puis **crop au bounding box France**

On remplit d'abord la grille complete, puis on **rogne au rectangle minimal**
contenant tous les pixels France — cela reduit significativement le padding ulterieur.

In [ ]:
dates_all   = np.sort(gdf['DATE'].unique())
T           = len(dates_all)
date_to_idx = {d: i for i, d in enumerate(dates_all)}
months_arr  = pd.DatetimeIndex(dates_all).month.values.astype(np.int32)
years_arr   = pd.DatetimeIndex(dates_all).year.values.astype(np.int32)

col_idx  = lambx_g.map(x_to_col).values.astype(np.int32)
row_idx  = lamby_g.map(y_to_row).values.astype(np.int32)
t_idx    = gdf['DATE'].map(date_to_idx).values.astype(np.int32)
swi_vals = gdf['SWI_UNIF_MENS'].values.astype(np.float32)

swi_full = np.full((T, H_full, W_full), np.nan, dtype=np.float32)
swi_full[t_idx, row_idx, col_idx] = swi_vals
gc.collect()
print(f'swi_full : {swi_full.shape}  ({swi_full.nbytes/1e6:.0f} MB)')

In [ ]:
# Bounding box des pixels France (masque)
land_full = np.isfinite(swi_full).any(axis=0)   # (H_full, W_full)
rows_ok = np.where(land_full.any(axis=1))[0]
cols_ok = np.where(land_full.any(axis=0))[0]
r0, r1 = int(rows_ok[0]), int(rows_ok[-1]) + 1
c0, c1 = int(cols_ok[0]), int(cols_ok[-1]) + 1

# Crop
swi_maps  = swi_full[:, r0:r1, c0:c1].copy()
land_mask = land_full[r0:r1, c0:c1]
del swi_full; gc.collect()

H, W = swi_maps.shape[1], swi_maps.shape[2]
print(f'Grille complete : {H_full} x {W_full}')
print(f'Bounding box France : {H} x {W}  (crop [{r0}:{r1}, {c0}:{c1}])')
print(f'Reduction : {(1 - H*W/(H_full*W_full))*100:.0f}% de cellules en moins')

## 4. Normalisation mensuelle + padding

In [ ]:
# Stats mensuelles
monthly_stats = {}
for m in range(1, 13):
    vals = swi_maps[months_arr == m][:, land_mask]
    monthly_stats[m] = (float(np.nanmean(vals)), float(np.nanstd(vals)))

# Normalisation
swi_norm = np.full_like(swi_maps, np.nan)
for m in range(1, 13):
    idx_m = months_arr == m
    mu_m, sig_m = monthly_stats[m]
    swi_norm[idx_m] = (swi_maps[idx_m] - mu_m) / (sig_m + 1e-8)
swi_norm_filled = np.nan_to_num(swi_norm, nan=0.0).astype(np.float32)

In [ ]:
# Padding au multiple de 8 le plus proche
H_pad = int(np.ceil(H / 8)) * 8
W_pad = int(np.ceil(W / 8)) * 8
pad_h, pad_w = H_pad - H, W_pad - W

swi_padded  = np.pad(swi_norm_filled, ((0,0),(0,pad_h),(0,pad_w)), constant_values=0.0)
mask_padded = np.pad(land_mask.astype(np.float32), ((0,pad_h),(0,pad_w)), constant_values=0.0)

print(f'Crop      : {H} x {W}')
print(f'Padde     : {H_pad} x {W_pad}  (pad_h={pad_h}, pad_w={pad_w})')
print(f'swi_padded : {swi_padded.shape}  ({swi_padded.nbytes/1e6:.0f} MB)')

## 5. Sauvegarde

In [ ]:
np.save(OUT_DIR / 'swi_maps.npy',    swi_maps)
np.save(OUT_DIR / 'swi_padded.npy',  swi_padded)
np.save(OUT_DIR / 'land_mask.npy',   land_mask)
np.save(OUT_DIR / 'mask_padded.npy', mask_padded)

meta = {
    'H': int(H), 'W': int(W), 'H_pad': int(H_pad), 'W_pad': int(W_pad),
    'H_full': int(H_full), 'W_full': int(W_full),
    'crop': {'r0':r0,'r1':r1,'c0':c0,'c1':c1},
    'STEP': STEP, 'T': int(T),
    'dates': [str(d) for d in dates_all],
    'months': months_arr.tolist(),
    'years':  years_arr.tolist(),
    'monthly_stats': {str(k): list(v) for k, v in monthly_stats.items()},
}
with open(OUT_DIR / 'metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Sauvegarde terminee dans', OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(f'  {p.name:<25} {p.stat().st_size/1e6:.1f} MB')